# Ungraded Lab: Unsupervised & Time Series Challenge Lab

## Task 1: Time Series Preparation and Analysis  

<b>Step 1:</b> Load and Prepare Time Series Data

In [ ]:
# Import required libraries
import pandas as pd
import numpy as np
from statsmodels.tsa.seasonal import seasonal_decompose
import matplotlib.pyplot as plt
from statsmodels.tsa.stattools import adfuller
from statsmodels.tsa.arima.model import ARIMA
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans

# Load dataset
df = pd.read_csv('smart_city_incidents_cleaned.csv')
df['timestamp'] = pd.to_datetime(df['timestamp'])

# Create daily issue counts
# Count how many issues happened each day
daily_issues = <YOUR CODE HERE>
daily_issues.columns = ['date', 'issue_count']
daily_issues.set_index('date', inplace=True)

<b>Step 2:</b> Analyze Temporal Patterns

In [ ]:
# Perform time series decomposition
decomposition = seasonal_decompose(daily_issues['issue_count'], period=7)

# Plot components
fig, (ax1, ax2, ax3) = plt.subplots(3, 1, figsize=(12, 10))
decomposition.trend.plot(ax=ax1, title='Trend')
decomposition.seasonal.plot(ax=ax2, title='Seasonality')
decomposition.resid.plot(ax=ax3, title='Residuals')
plt.tight_layout()
plt.show()

Review your time series patterns:
- Identify peak reporting days
- Note any seasonal patterns
- Check for any unusual spikes in incidents

## Task 2: Incident Clustering and Profiling

<b>Step 1:</b> Prepare Features for Clustering

In [ ]:
# Select and prepare features
clustering_features = [
    'time_to_resolution', 'severity_score',
    'num_previous_reports', 'issue_complexity',
    'response_team_rating'
]

# Scale features by creating a scaler to standardize the data

X = df[clustering_features]
scaler = <YOUR CODE HERE>
# Fit the scaler and transform the data
X_scaled = <YOUR CODE HERE>

<b>Step 2:</b> Implement K-means Clustering

In [ ]:
# Find optimal number of clusters
inertias = []
for k in range(1, 11):
    # Create a KMeans model with k clusters
    <YOUR CODE HERE>
    # Fit the model to the scaled data
    <YOUR CODE HERE>
    # Save the inertia (how well the clusters fit) for this k
    <YOUR CODE HERE>

# Plot elbow curve
<YOUR CODE HERE>


# Apply clustering with optimal k
optimal_k = 4  # Adjust based on elbow curve
kmeans = KMeans(n_clusters=optimal_k, random_state=42)
df['cluster'] = kmeans.fit_predict(X_scaled)

Review your cluster analysis:
- Review cluster characteristics
- Analyze issue types within each cluster
- Note patterns in resolution times

## Task 3: Combined Analysis and Forecasting

<b>Step 1:</b> Analyze Clusters Across Time

In [ ]:
# Create time-based cluster analysis
cluster_time_analysis = (
    df.groupby([df['timestamp'].dt.date, 'cluster'])
      .size()
      .unstack(fill_value=0)  # fill missing combinations with 0s
)

# Plot cluster volumes over time
plt.figure(figsize=(12, 6))
cluster_time_analysis.plot()
plt.title('Cluster Volumes Over Time')
plt.xlabel('Date')
plt.ylabel('Number of Issues')
plt.legend(['Cluster ' + str(i) for i in range(optimal_k)])
plt.show()

<b>Step 2:</b> Build Forecasting Model

In [ ]:
# Check if time series is stationary
def check_stationarity(timeseries):
    print("Running Augmented Dickey-Fuller test...")
    result = adfuller(timeseries.dropna())
    print(f"ADF Statistic: {result[0]:.4f}")
    print(f"p-value: {result[1]:.4f}")
    for key, value in result[4].items():
        print(f"Critical Value ({key}): {value:.4f}")
    if result[1] < 0.05:
        print("=> The series is stationary (reject H0)")
    else:
        print("=> The series is NOT stationary (fail to reject H0)")

# Adjust layout for any previous plots
plt.tight_layout()
plt.show()

# Run stationarity test for each cluster time series
for cluster in range(optimal_k):
    series = cluster_time_analysis[cluster]
    print(f"\nCluster {cluster} stationarity test:")
    check_stationarity(series)

# Prepare data for each cluster
def forecast_cluster_volume(cluster_data, steps=7, freq='D'):
    cluster_data.index = pd.to_datetime(cluster_data.index)
    cluster_data = cluster_data.asfreq(freq).ffill()
    model = ARIMA(cluster_data, order=(1,1,1))
    results = model.fit()
    forecast = results.forecast(steps=steps)
    return forecast

# Generate forecasts for each cluster's future issue volume
forecasts = {}
for cluster in range(optimal_k):
    cluster_data = cluster_time_analysis[cluster]
    forecasts[cluster] = forecast_cluster_volume(cluster_data)

Review your combined analysis:
- Compare cluster trends over time
- Identify which clusters show seasonal patterns
- Review forecast accuracy for different clusters

## Task 4: Insights and Recommendations

<b>Step 1:</b> Calculate Key Metrics

In [ ]:
# Calculate the average value of each feature for every cluster
cluster_profiles = <YOUR CODE HERE>

# Calculate the average resolution times by cluster and issue type
resolution_analysis = <YOUR CODE HERE>

# Generate summary statistics
print("\nCluster Profiles:")
print(cluster_profiles)
print("\nResolution Times by Issue Type and Cluster:")
print(resolution_analysis)

<b>Step 2:</b> Visualize Key Findings

In [ ]:
# Calculate average resolution time by cluster and issue type, then pivot for easier visualization
resolution_analysis = df.groupby(['cluster', 'issue_type'])['time_to_resolution'].mean().unstack()

# Display summary statistics
print("\nCluster Profiles (Mean Feature Values per Cluster):")
print(cluster_profiles)
print("\nAverage Resolution Times by Issue Type and Cluster:")
print(resolution_analysis)
# Create visualization of key metrics in a 2x2 grid for easy comparison
fig, axes = plt.subplots(2, 2, figsize=(15, 10))

# Plot 1: Average resolution time by cluster
# - Helps understand which clusters take longer to resolve issues
# - Useful for prioritizing support or resources
cluster_profiles['time_to_resolution'].plot(kind='bar', ax=axes[0,0], color='skyblue')
axes[0,0].set_title('Average Resolution Time by Cluster')
axes[0,0].set_ylabel('Time to Resolution (hours/days)')
axes[0,0].set_xlabel('Cluster')
axes[0,0].grid(axis='y')

# Plot 2: Severity score distribution by cluster (boxplot)
# - Shows spread and outliers of severity scores in each cluster
# - Helps identify if some clusters tend to have more severe issues
df.boxplot(column='severity_score', by='cluster', ax=axes[0,1])
axes[0,1].set_title('Severity Score Distribution by Cluster')
axes[0,1].set_ylabel('Severity Score')
axes[0,1].set_xlabel('Cluster')
plt.suptitle('') # Removes default suptitle from boxplot

# Plot 3: Distribution of issue types (pie chart)
# - Overall share of each issue type across all clusters
# - Provides insight into the most common types of issues faced
df['issue_type'].value_counts().plot(kind='pie', ax=axes[1,0], autopct='%1.1f%%', startangle=140)
axes[1,0].set_title('Distribution of Issue Types')
axes[1,0].set_ylabel('')

# Plot 4: Forecasted volume by cluster (line plot)
# - Shows expected volume of issues for each cluster over the forecast period
# - Useful for planning staffing or support levels proactively
pd.DataFrame(forecasts).plot(ax=axes[1,1], marker='o')
axes[1,1].set_title('Forecasted Volumes by Cluster')
axes[1,1].set_xlabel('Days Ahead')
axes[1,1].set_ylabel('Forecasted Issue Volume')
axes[1,1].grid(True)
plt.tight_layout()
plt.show()

Review your final results:
- Identify highest priority clusters
- Note patterns in issue resolution
- List actionable recommendations

## Solution Code
Need a hand or are curious to compare your approach? Below is a complete solution you can use as a reference. This is just one of many valid ways to solve the problem. Make sure to give it a try on your own first. Use this implementation to troubleshoot, learn new techniques, or confirm your logic. Keep experimenting and enjoy the process!

## Task 1: Time Series Preparation and Analysis Solution Code

<b>Step 1:</b> Load and Prepare Time Series Data

In [ ]:
# Import required libraries
import pandas as pd
import numpy as np
from statsmodels.tsa.seasonal import seasonal_decompose
import matplotlib.pyplot as plt
from statsmodels.tsa.stattools import adfuller
from statsmodels.tsa.arima.model import ARIMA
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans

# Load dataset
df = pd.read_csv('smart_city_incidents_cleaned.csv')
df['timestamp'] = pd.to_datetime(df['timestamp'])

# Create daily issue counts
# Count how many issues happened each day
daily_issues = df.groupby(df['timestamp'].dt.date).size().reset_index()
daily_issues.columns = ['date', 'issue_count']
daily_issues.set_index('date', inplace=True)

<b>Step 2:</b> Analyze Temporal Patterns

In [ ]:
# Perform time series decomposition
decomposition = seasonal_decompose(daily_issues['issue_count'], period=7)

# Plot components
fig, (ax1, ax2, ax3) = plt.subplots(3, 1, figsize=(12, 10))
decomposition.trend.plot(ax=ax1, title='Trend')
decomposition.seasonal.plot(ax=ax2, title='Seasonality')
decomposition.resid.plot(ax=ax3, title='Residuals')
plt.tight_layout()
plt.show()

## Task 2: Incident Clustering and Profiling Solution Code

<b>Step 1:</b> Prepare Features for Clustering

In [ ]:
# Select and prepare features
clustering_features = [
    'time_to_resolution', 'severity_score',
    'num_previous_reports', 'issue_complexity',
    'response_team_rating'
]

# Scale features by creating a scaler to standardize the data
X = df[clustering_features]
scaler = StandardScaler()
# Fit the scaler and transform the data
X_scaled = scaler.fit_transform(X)

<b>Step 2:</b> Implement K-means Clustering

In [ ]:
# Find optimal number of clusters
inertias = []
for k in range(1, 11):
    # Create a KMeans model with k clusters
    kmeans = KMeans(n_clusters=k, random_state=42)
    # Fit the model to the scaled data
    kmeans.fit(X_scaled)
    # Save the inertia (how well the clusters fit) for this k
    inertias.append(kmeans.inertia_)

# Plot elbow curve
plt.plot(range(1, 11), inertias, 'bx-')
plt.xlabel('k')
plt.ylabel('Inertia')
plt.title('Elbow Method for Optimal k')
plt.show()

# Apply clustering with optimal k
optimal_k = 4  # Adjust based on elbow curve
kmeans = KMeans(n_clusters=optimal_k, random_state=42)
df['cluster'] = kmeans.fit_predict(X_scaled)

## Task 3: Combined Analysis and Forecasting Solution Code

<b>Step 1:</b> Analyze Clusters Across Time

In [ ]:
# Create time-based cluster analysis
cluster_time_analysis = (
    df.groupby([df['timestamp'].dt.date, 'cluster'])
      .size()
      .unstack(fill_value=0)  # fill missing combinations with 0s
)

# Plot cluster volumes over time
plt.figure(figsize=(12, 6))
cluster_time_analysis.plot()
plt.title('Cluster Volumes Over Time')
plt.xlabel('Date')
plt.ylabel('Number of Issues')
plt.legend(['Cluster ' + str(i) for i in range(optimal_k)])
plt.show()

<b>Step 2:</b> Build Forecasting Model

In [ ]:
# Check if time series is stationary
def check_stationarity(timeseries):
    print("Running Augmented Dickey-Fuller test...")
    result = adfuller(timeseries.dropna())
    print(f"ADF Statistic: {result[0]:.4f}")
    print(f"p-value: {result[1]:.4f}")
    for key, value in result[4].items():
        print(f"Critical Value ({key}): {value:.4f}")
    if result[1] < 0.05:
        print("=> The series is stationary (reject H0)")
    else:
        print("=> The series is NOT stationary (fail to reject H0)")

# Adjust layout for any previous plots
plt.tight_layout()
plt.show()

# Run stationarity test for each cluster time series
for cluster in range(optimal_k):
    series = cluster_time_analysis[cluster]
    print(f"\nCluster {cluster} stationarity test:")
    check_stationarity(series)


# Prepare data for each cluster
def forecast_cluster_volume(cluster_data, steps=7, freq='D'):
    cluster_data.index = pd.to_datetime(cluster_data.index)
    cluster_data = cluster_data.asfreq(freq).ffill()
    model = ARIMA(cluster_data, order=(1,1,1))
    results = model.fit()
    forecast = results.forecast(steps=steps)
    return forecast


# Generate forecasts for each cluster's future issue volume
forecasts = {}
for cluster in range(optimal_k):
    cluster_data = cluster_time_analysis[cluster]
    forecasts[cluster] = forecast_cluster_volume(cluster_data)

## Task 4: Insights and Recommendations Solution Code

<b>Step 1:</b> Calculate Key Metrics

In [ ]:
# Calculate the average value of each feature for every clusterCalculate cluster profiles
cluster_profiles = df.groupby('cluster')[clustering_features].mean()

# Calculate the average resolution times by cluster and issue type
resolution_analysis = df.groupby(['cluster', 'issue_type'])['time_to_resolution'].mean().unstack()

# Generate summary statistics
print("\nCluster Profiles:")
print(cluster_profiles)
print("\nResolution Times by Issue Type and Cluster:")
print(resolution_analysis)

<b>Step 2:</b> Visualize Key Findings

In [ ]:
# Calculate average resolution time by cluster and issue type, then pivot for easier visualization
resolution_analysis = df.groupby(['cluster', 'issue_type'])['time_to_resolution'].mean().unstack()

# Display summary statistics
print("\nCluster Profiles (Mean Feature Values per Cluster):")
print(cluster_profiles)
print("\nAverage Resolution Times by Issue Type and Cluster:")
print(resolution_analysis)
# Create visualization of key metrics in a 2x2 grid for easy comparison
fig, axes = plt.subplots(2, 2, figsize=(15, 10))

# Plot 1: Average resolution time by cluster
# - Helps understand which clusters take longer to resolve issues
# - Useful for prioritizing support or resources
cluster_profiles['time_to_resolution'].plot(kind='bar', ax=axes[0,0], color='skyblue')
axes[0,0].set_title('Average Resolution Time by Cluster')
axes[0,0].set_ylabel('Time to Resolution (hours/days)')
axes[0,0].set_xlabel('Cluster')
axes[0,0].grid(axis='y')

# Plot 2: Severity score distribution by cluster (boxplot)
# - Shows spread and outliers of severity scores in each cluster
# - Helps identify if some clusters tend to have more severe issues
df.boxplot(column='severity_score', by='cluster', ax=axes[0,1])
axes[0,1].set_title('Severity Score Distribution by Cluster')
axes[0,1].set_ylabel('Severity Score')
axes[0,1].set_xlabel('Cluster')
plt.suptitle('') # Removes default suptitle from boxplot

# Plot 3: Distribution of issue types (pie chart)
# - Overall share of each issue type across all clusters
# - Provides insight into the most common types of issues faced
df['issue_type'].value_counts().plot(kind='pie', ax=axes[1,0], autopct='%1.1f%%', startangle=140)
axes[1,0].set_title('Distribution of Issue Types')
axes[1,0].set_ylabel('')

# Plot 4: Forecasted volume by cluster (line plot)
# - Shows expected volume of issues for each cluster over the forecast period
# - Useful for planning staffing or support levels proactively
pd.DataFrame(forecasts).plot(ax=axes[1,1], marker='o')
axes[1,1].set_title('Forecasted Volumes by Cluster')
axes[1,1].set_xlabel('Days Ahead')
axes[1,1].set_ylabel('Forecasted Issue Volume')
axes[1,1].grid(True)
plt.tight_layout()
plt.show()